In [1]:


import os
os.environ["HADOOP_HOME"] = "E://spark//spark-3.5.2-bin-hadoop3"


# Q1. Load all the files using SparkSession. (PySpark -Dataframe)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import to_date, datediff
from pyspark.sql.window import Window
import pandas as pd

spark = SparkSession.builder.appName("PySpark_Assignment").getOrCreate()



In [3]:
market_fact_df = spark.read.load("market_fact.csv", format = "csv", sep = ",", inferSchema = True, header = True)

cust_dimen_df = spark.read.load("cust_dimen.csv", format = "csv", sep = ",", inferSchema = True, header = True)

orders_dimen_df = spark.read.load("orders_dimen.csv", format = "csv", sep = ",", inferSchema = True, header = True)

prod_dimen_df = spark.read.load("prod_dimen.csv", format = "csv", sep = ",", inferSchema = True, header = True)

shipping_dimen_df = spark.read.load("shipping_dimen.csv", format = "csv", sep = ",", inferSchema = True, header = True)



In [4]:
market_fact_df.show()

+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+
|  Ord_id|Prod_id| Ship_id|  Cust_id|    Sales|Discount|Order_Quantity| Profit|Shipping_Cost|Product_Base_Margin|
+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+
|Ord_5446|Prod_16|SHP_7609|Cust_1818|   136.81|    0.01|            23| -30.51|          3.6|               0.56|
|Ord_5406|Prod_13|SHP_7549|Cust_1818|    42.27|    0.01|            13|   4.56|         0.93|               0.54|
|Ord_5446| Prod_4|SHP_7610|Cust_1818|  4701.69|     0.0|            26| 1148.9|          2.5|               0.59|
|Ord_5456| Prod_6|SHP_7625|Cust_1818|  2337.89|    0.09|            43| 729.34|         14.3|               0.37|
|Ord_5485|Prod_17|SHP_7664|Cust_1818|  4233.15|    0.08|            35|1219.87|         26.3|               0.38|
|Ord_5446| Prod_6|SHP_7608|Cust_1818|   164.02|    0.03|            23| -47.64|         

In [5]:
cust_dimen_df.show()

+------------------+--------+-------+----------------+-------+
|     Customer_Name|Province| Region|Customer_Segment|Cust_id|
+------------------+--------+-------+----------------+-------+
|MUHAMMED MACINTYRE| NUNAVUT|NUNAVUT|  SMALL BUSINESS| Cust_1|
|      BARRY FRENCH| NUNAVUT|NUNAVUT|        CONSUMER| Cust_2|
|     CLAY ROZENDAL| NUNAVUT|NUNAVUT|       CORPORATE| Cust_3|
|    CARLOS SOLTERO| NUNAVUT|NUNAVUT|        CONSUMER| Cust_4|
|      CARL JACKSON| NUNAVUT|NUNAVUT|       CORPORATE| Cust_5|
|    MONICA FEDERLE| NUNAVUT|NUNAVUT|       CORPORATE| Cust_6|
|   DOROTHY BADDERS| NUNAVUT|NUNAVUT|     HOME OFFICE| Cust_7|
|   NEOLA SCHNEIDER| NUNAVUT|NUNAVUT|     HOME OFFICE| Cust_8|
|       CARLOS DALY| NUNAVUT|NUNAVUT|     HOME OFFICE| Cust_9|
|     CLAUDIA MINER| NUNAVUT|NUNAVUT|  SMALL BUSINESS|Cust_10|
|  ALLEN ROSENBLATT| NUNAVUT|NUNAVUT|  SMALL BUSINESS|Cust_11|
|   SYLVIA FOULSTON| NUNAVUT|NUNAVUT|     HOME OFFICE|Cust_12|
|       JIM RADFORD| NUNAVUT|NUNAVUT|       CORPORATE|C

In [6]:
orders_dimen_df.show()

+--------+----------+--------------+------+
|Order_ID|Order_Date|Order_Priority|Ord_id|
+--------+----------+--------------+------+
|       3|13-10-2010|           LOW| Ord_1|
|     293|01-10-2012|          HIGH| Ord_2|
|     483|10-07-2011|          HIGH| Ord_3|
|     515|28-08-2010| NOT SPECIFIED| Ord_4|
|     613|17-06-2011|          HIGH| Ord_5|
|     643|24-03-2011|          HIGH| Ord_6|
|     678|26-02-2010|           LOW| Ord_7|
|     807|23-11-2010|        MEDIUM| Ord_8|
|     868|08-06-2012| NOT SPECIFIED| Ord_9|
|     933|04-08-2012| NOT SPECIFIED|Ord_10|
|     995|30-05-2011|        MEDIUM|Ord_11|
|     998|25-11-2009| NOT SPECIFIED|Ord_12|
|    1154|14-02-2012|      CRITICAL|Ord_13|
|    1344|15-04-2012|           LOW|Ord_14|
|    1412|12-03-2010| NOT SPECIFIED|Ord_15|
|    1539|09-03-2011|           LOW|Ord_16|
|    1540|04-08-2012|          HIGH|Ord_17|
|    1702|06-05-2011|          HIGH|Ord_18|
|    1761|23-12-2010|          HIGH|Ord_19|
|    1792|08-11-2010|           

In [7]:
prod_dimen_df.show()

+----------------+--------------------+-------+
|Product_Category|Product_Sub_Category|Prod_id|
+----------------+--------------------+-------+
| OFFICE SUPPLIES|STORAGE & ORGANIZ...| Prod_1|
| OFFICE SUPPLIES|          APPLIANCES| Prod_2|
| OFFICE SUPPLIES|BINDERS AND BINDE...| Prod_3|
|      TECHNOLOGY|TELEPHONES AND CO...| Prod_4|
|       FURNITURE|  OFFICE FURNISHINGS| Prod_5|
| OFFICE SUPPLIES|               PAPER| Prod_6|
| OFFICE SUPPLIES|        RUBBER BANDS| Prod_7|
|      TECHNOLOGY|COMPUTER PERIPHERALS| Prod_8|
| OFFICE SUPPLIES|           ENVELOPES| Prod_9|
|       FURNITURE|           BOOKCASES|Prod_10|
|       FURNITURE|              TABLES|Prod_11|
| OFFICE SUPPLIES|              LABELS|Prod_12|
| OFFICE SUPPLIES| PENS & ART SUPPLIES|Prod_13|
|      TECHNOLOGY|     COPIERS AND FAX|Prod_14|
|       FURNITURE|  CHAIRS & CHAIRMATS|Prod_15|
| OFFICE SUPPLIES|SCISSORS, RULERS ...|Prod_16|
|      TECHNOLOGY|     OFFICE MACHINES|Prod_17|
+----------------+--------------------+-

In [8]:
shipping_dimen_df.show()

+--------+--------------+----------+-------+
|Order_ID|     Ship_Mode| Ship_Date|Ship_id|
+--------+--------------+----------+-------+
|       3|   REGULAR AIR|20-10-2010|  SHP_1|
|     293|DELIVERY TRUCK|02-10-2012|  SHP_2|
|     293|   REGULAR AIR|03-10-2012|  SHP_3|
|     483|   REGULAR AIR|12-07-2011|  SHP_4|
|     515|   REGULAR AIR|30-08-2010|  SHP_5|
|     613|   REGULAR AIR|17-06-2011|  SHP_6|
|     613|   REGULAR AIR|18-06-2011|  SHP_7|
|     643|   EXPRESS AIR|25-03-2011|  SHP_8|
|     678|   REGULAR AIR|26-02-2010|  SHP_9|
|     807|   REGULAR AIR|24-11-2010| SHP_10|
|     868|   REGULAR AIR|09-06-2012| SHP_11|
|     868|   REGULAR AIR|10-06-2012| SHP_12|
|     933|   REGULAR AIR|04-08-2012| SHP_13|
|     995|   REGULAR AIR|31-05-2011| SHP_14|
|     998|   REGULAR AIR|26-11-2009| SHP_15|
|    1154|DELIVERY TRUCK|16-02-2012| SHP_16|
|    1154|   REGULAR AIR|16-02-2012| SHP_17|
|    1344|   REGULAR AIR|22-04-2012| SHP_18|
|    1344|   REGULAR AIR|19-04-2012| SHP_19|
|    1412|

# Q2. Join all the Data frames and create a new Data frame called Full_DataFrame in such a way that the new data frame does not contain duplicate columns. (cust_dimen, market_fact, orders_dimen, prod_dimen, shipping_dimen)

In [9]:
# Way-1

df = market_fact_df.join(cust_dimen_df, on="Cust_id", how='left')
df = df.join(orders_dimen_df, on="Ord_id", how='left')
df = df.join(prod_dimen_df, on="Prod_id", how='left')
df = df.join(shipping_dimen_df[["Ship_Mode", "Ship_Date", "Ship_id"]], on="Ship_id", how='left')
df.show()

+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+
| Ship_id|Prod_id|  Ord_id|  Cust_id|    Sales|Discount|Order_Quantity| Profit|Shipping_Cost|Product_Base_Margin| Customer_Name|        Province|  Region|Customer_Segment|Order_ID|Order_Date|Order_Priority|Product_Category|Product_Sub_Category|     Ship_Mode| Ship_Date|
+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+
|SHP_7609|Prod_16|Ord_5446|Cust_1818|   136.81|    0.01|            23| -30.51|          3.6|               0.56| AARON BERGMAN|         ALBERTA|    WEST|       CORPORATE|   36262|27-07-2

In [10]:
'''

# Convert the list of columns to a set to remove duplicates
unique_columns = list(set(df.columns))

# Select the unique columns from the DataFrame
df = df.select(*unique_columns)
'''

'\n\n# Convert the list of columns to a set to remove duplicates\nunique_columns = list(set(df.columns))\n\n# Select the unique columns from the DataFrame\ndf = df.select(*unique_columns)\n'

In [11]:
# Way -2 - using Pyspark SQL


market_fact_df.createOrReplaceTempView("market_fact_view")

cust_dimen_df.createOrReplaceTempView("cust_dimen_view")

orders_dimen_df.createOrReplaceTempView("orders_dimen_view")

prod_dimen_df.createOrReplaceTempView("prod_dimen_view")

shipping_dimen_df.createOrReplaceTempView("shipping_dimen_view")

df4 = spark.sql(""" 
Select market.*, customer.*, orders.*, product.*, shipping.* 
from market_fact_view market left join cust_dimen_view customer 
on market.cust_id = customer.cust_id
left join orders_dimen_view orders 
on market.ord_id = orders.ord_id
left join prod_dimen_view product
on market.prod_id = product.prod_id
left join shipping_dimen_view shipping
on market.ship_id = shipping.ship_id
""")

In [12]:
# Convert the list of columns to a set to remove duplicates
unique_columns = list(set(df4.columns))

# Select the unique columns from the DataFrame
df4 = df.select(*unique_columns)

df4.show()

+-------+--------------+--------------+-------+--------+-------------+--------------+--------+---------+--------------+----------------+----------+----------------+----------------+--------+-------------------+--------+--------------------+----------+--------+---------+
| Profit|Order_Quantity|     Ship_Mode|Prod_id| Ship_id|Shipping_Cost|Order_Priority|Order_ID|  Cust_id| Customer_Name|Customer_Segment| Ship_Date|Product_Category|        Province|Discount|Product_Base_Margin|  Ord_id|Product_Sub_Category|Order_Date|  Region|    Sales|
+-------+--------------+--------------+-------+--------+-------------+--------------+--------+---------+--------------+----------------+----------+----------------+----------------+--------+-------------------+--------+--------------------+----------+--------+---------+
| -30.51|            23|   REGULAR AIR|Prod_16|SHP_7609|          3.6| NOT SPECIFIED|   36262|Cust_1818| AARON BERGMAN|       CORPORATE|28-07-2010| OFFICE SUPPLIES|         ALBERTA|    0.

# Q3. Convert the Order_Date and Ship_Date columns type into Date type. And print the schema and show the top 5 records for Order_Date and Ship_Date columns

In [13]:
df.printSchema()

root
 |-- Ship_id: string (nullable = true)
 |-- Prod_id: string (nullable = true)
 |-- Ord_id: string (nullable = true)
 |-- Cust_id: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Order_Quantity: integer (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- Product_Base_Margin: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Province: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Customer_Segment: string (nullable = true)
 |-- Order_ID: integer (nullable = true)
 |-- Order_Date: string (nullable = true)
 |-- Order_Priority: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_Sub_Category: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Ship_Date: string (nullable = true)



In [14]:
df = df.withColumn("Order_Date", to_date("Order_Date", "dd-MM-yyyy"))
df = df.withColumn("Ship_Date", to_date("Ship_Date", "dd-MM-yyyy"))

df.show()

+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+
| Ship_id|Prod_id|  Ord_id|  Cust_id|    Sales|Discount|Order_Quantity| Profit|Shipping_Cost|Product_Base_Margin| Customer_Name|        Province|  Region|Customer_Segment|Order_ID|Order_Date|Order_Priority|Product_Category|Product_Sub_Category|     Ship_Mode| Ship_Date|
+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+
|SHP_7609|Prod_16|Ord_5446|Cust_1818|   136.81|    0.01|            23| -30.51|          3.6|               0.56| AARON BERGMAN|         ALBERTA|    WEST|       CORPORATE|   36262|2010-07

In [15]:
df.printSchema()

root
 |-- Ship_id: string (nullable = true)
 |-- Prod_id: string (nullable = true)
 |-- Ord_id: string (nullable = true)
 |-- Cust_id: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Order_Quantity: integer (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- Product_Base_Margin: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Province: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Customer_Segment: string (nullable = true)
 |-- Order_ID: integer (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Order_Priority: string (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Product_Sub_Category: string (nullable = true)
 |-- Ship_Mode: string (nullable = true)
 |-- Ship_Date: date (nullable = true)



In [16]:
df[["Order_Date", "Ship_Date"]].show(5)

+----------+----------+
|Order_Date| Ship_Date|
+----------+----------+
|2010-07-27|2010-07-28|
|2009-07-07|2009-07-08|
|2010-07-27|2010-07-27|
|2010-11-09|2010-11-11|
|2009-07-01|2009-07-08|
+----------+----------+
only showing top 5 rows



# Q4. Find the top 3 customers who have the maximum number of orders.

In [17]:
df.show()

+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+
| Ship_id|Prod_id|  Ord_id|  Cust_id|    Sales|Discount|Order_Quantity| Profit|Shipping_Cost|Product_Base_Margin| Customer_Name|        Province|  Region|Customer_Segment|Order_ID|Order_Date|Order_Priority|Product_Category|Product_Sub_Category|     Ship_Mode| Ship_Date|
+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+
|SHP_7609|Prod_16|Ord_5446|Cust_1818|   136.81|    0.01|            23| -30.51|          3.6|               0.56| AARON BERGMAN|         ALBERTA|    WEST|       CORPORATE|   36262|2010-07

In [18]:
# Way -1

df.createOrReplaceTempView("joined_data_view")

spark.sql("Select Cust_id, Customer_Name, Customer_Segment, count(distinct Ord_id) from joined_data_view group by 1,2,3 order by 4 desc").show(3)

+---------+-----------------+----------------+----------------------+
|  Cust_id|    Customer_Name|Customer_Segment|count(DISTINCT Ord_id)|
+---------+-----------------+----------------+----------------------+
|Cust_1140|    PATRICK JONES|     HOME OFFICE|                    17|
| Cust_576|MICHAEL DOMINGUEZ|     HOME OFFICE|                    16|
| Cust_999|    SALLY HUGHSBY|        CONSUMER|                    15|
+---------+-----------------+----------------+----------------------+
only showing top 3 rows



In [19]:
# Way 2

df.groupBy(df.Cust_id, df.Customer_Name, df.Customer_Segment).agg(F.countDistinct("Ord_id").alias("Order_count")).sort(F.col("order_count").desc()).show(3)

+---------+-----------------+----------------+-----------+
|  Cust_id|    Customer_Name|Customer_Segment|Order_count|
+---------+-----------------+----------------+-----------+
|Cust_1140|    PATRICK JONES|     HOME OFFICE|         17|
| Cust_576|MICHAEL DOMINGUEZ|     HOME OFFICE|         16|
| Cust_999|    SALLY HUGHSBY|        CONSUMER|         15|
+---------+-----------------+----------------+-----------+
only showing top 3 rows



# Q5. Create a new column DaysTakenForDelivery that contains the date difference between Order_Date and Ship_Date.

In [20]:
# Way - 1

In [21]:
df = df.withColumn("DaysTakenForDelivery", datediff("Ship_Date", "Order_Date"))
df.show()

+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+--------------------+
| Ship_id|Prod_id|  Ord_id|  Cust_id|    Sales|Discount|Order_Quantity| Profit|Shipping_Cost|Product_Base_Margin| Customer_Name|        Province|  Region|Customer_Segment|Order_ID|Order_Date|Order_Priority|Product_Category|Product_Sub_Category|     Ship_Mode| Ship_Date|DaysTakenForDelivery|
+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+--------------------+
|SHP_7609|Prod_16|Ord_5446|Cust_1818|   136.81|    0.01|            23| -30.51|          3.6|               0.56| AARON BERG

In [22]:
df[["Ship_Date", "Order_Date", "DaysTakenForDelivery"]].show(100)

+----------+----------+--------------------+
| Ship_Date|Order_Date|DaysTakenForDelivery|
+----------+----------+--------------------+
|2010-07-28|2010-07-27|                   1|
|2009-07-08|2009-07-07|                   1|
|2010-07-27|2010-07-27|                   0|
|2010-11-11|2010-11-09|                   2|
|2009-07-08|2009-07-01|                   7|
|2010-07-28|2010-07-27|                   1|
|2011-05-30|2011-05-28|                   2|
|2011-12-31|2011-12-29|                   2|
|2011-12-31|2011-12-29|                   2|
|2011-12-31|2011-12-29|                   2|
|2010-05-26|2010-05-26|                   0|
|2011-10-31|2011-10-30|                   1|
|2011-02-26|2011-02-24|                   2|
|2011-12-27|2011-12-25|                   2|
|2011-12-26|2011-12-25|                   1|
|2009-08-17|2009-08-15|                   2|
|2010-10-05|2010-10-04|                   1|
|2009-12-15|2009-12-13|                   2|
|2011-05-21|2011-05-12|                   9|
|2009-05-1

In [23]:
df.filter(df["Ord_id"] == 'Ord_2396')[["Ship_Date", "Order_Date", "DaysTakenForDelivery"]].show()

+----------+----------+--------------------+
| Ship_Date|Order_Date|DaysTakenForDelivery|
+----------+----------+--------------------+
|2011-01-26|2010-12-26|                  31|
|2011-01-22|2010-12-26|                  27|
|2011-01-14|2010-12-26|                  19|
+----------+----------+--------------------+



In [24]:
# Way - 2

df.createOrReplaceTempView("joined_data_view")

spark.sql("Select data.*,  Ship_Date - Order_Date DaysTakenForDelivery from joined_data_view data ").show()


+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+--------------------+--------------------+
| Ship_id|Prod_id|  Ord_id|  Cust_id|    Sales|Discount|Order_Quantity| Profit|Shipping_Cost|Product_Base_Margin| Customer_Name|        Province|  Region|Customer_Segment|Order_ID|Order_Date|Order_Priority|Product_Category|Product_Sub_Category|     Ship_Mode| Ship_Date|DaysTakenForDelivery|DaysTakenForDelivery|
+--------+-------+--------+---------+---------+--------+--------------+-------+-------------+-------------------+--------------+----------------+--------+----------------+--------+----------+--------------+----------------+--------------------+--------------+----------+--------------------+--------------------+
|SHP_7609|Prod_16|Ord_5446|Cust_1818|   136.81|    0.01|     

# Q6. Find the customer whose order took the maximum time to get delivered.

In [25]:
# Way - 1

df.select("Cust_id", "Customer_Name", "Customer_Segment", "DaysTakenForDelivery").sort(F.col("DaysTakenForDelivery").desc()).show()



+---------+-----------------+----------------+--------------------+
|  Cust_id|    Customer_Name|Customer_Segment|DaysTakenForDelivery|
+---------+-----------------+----------------+--------------------+
|Cust_1460|      DEAN PERCER|     HOME OFFICE|                  92|
| Cust_304|     ART FERGUSON|       CORPORATE|                  84|
| Cust_814|      TOM STIVERS|       CORPORATE|                  31|
|Cust_1481| CHARLES CRESTANI|        CONSUMER|                  28|
| Cust_814|      TOM STIVERS|       CORPORATE|                  27|
|Cust_1553|     MARIS LAWARE|     HOME OFFICE|                  24|
|Cust_1412|    MARIBETH DONA|     HOME OFFICE|                  22|
|Cust_1181|SUSAN MACKENDRICK|       CORPORATE|                  19|
|Cust_1311|      TOM STIVERS|       CORPORATE|                  19|
|Cust_1181|SUSAN MACKENDRICK|       CORPORATE|                  18|
|Cust_1481| CHARLES CRESTANI|        CONSUMER|                  17|
| Cust_934| LOGAN HAUSHALTER|       CORPORATE|  

In [26]:
df.select("Cust_id", "Customer_Name", "Customer_Segment", "DaysTakenForDelivery").sort(F.col("DaysTakenForDelivery").desc()).show(1)

+---------+-------------+----------------+--------------------+
|  Cust_id|Customer_Name|Customer_Segment|DaysTakenForDelivery|
+---------+-------------+----------------+--------------------+
|Cust_1460|  DEAN PERCER|     HOME OFFICE|                  92|
+---------+-------------+----------------+--------------------+
only showing top 1 row



In [27]:
# Way - 2

df.createOrReplaceTempView("joined_data_view")
spark.sql("Select Cust_id, Customer_Name, Customer_Segment, DaysTakenForDelivery from joined_data_view order by 4 desc").show(1)

+---------+-------------+----------------+--------------------+
|  Cust_id|Customer_Name|Customer_Segment|DaysTakenForDelivery|
+---------+-------------+----------------+--------------------+
|Cust_1460|  DEAN PERCER|     HOME OFFICE|                  92|
+---------+-------------+----------------+--------------------+
only showing top 1 row



# Q7. Using the windows function, retrieve total sales made by each product from the data.

In [28]:
# Way - 1

# Define the window specification
windowSpec = Window.partitionBy("Prod_id")

df.withColumn("Total_Sale_of_Product", F.sum("Sales").over(windowSpec)) \
  .select("Prod_id", "Product_Category", "Product_Sub_Category", "Total_Sale_of_Product") \
  .distinct().show()

+-------+----------------+--------------------+---------------------+
|Prod_id|Product_Category|Product_Sub_Category|Total_Sale_of_Product|
+-------+----------------+--------------------+---------------------+
| Prod_1| OFFICE SUPPLIES|STORAGE & ORGANIZ...|   1070182.6000000006|
|Prod_10|       FURNITURE|           BOOKCASES|    822652.0400000003|
|Prod_11|       FURNITURE|              TABLES|   1896008.1420000014|
|Prod_12| OFFICE SUPPLIES|              LABELS|    38981.55000000002|
|Prod_13| OFFICE SUPPLIES| PENS & ART SUPPLIES|   167107.21999999997|
|Prod_14|      TECHNOLOGY|     COPIERS AND FAX|   1130361.2999999996|
|Prod_15|       FURNITURE|  CHAIRS & CHAIRMATS|           1761836.55|
|Prod_16| OFFICE SUPPLIES|SCISSORS, RULERS ...|    80996.30999999997|
|Prod_17|      TECHNOLOGY|     OFFICE MACHINES|   2168697.1400000006|
| Prod_2| OFFICE SUPPLIES|          APPLIANCES|    736991.5399999997|
| Prod_3| OFFICE SUPPLIES|BINDERS AND BINDE...|   1022957.5900000007|
| Prod_4|      TECHN

In [29]:
# Way - 2

spark.sql("Select Prod_id, Product_Category, Product_Sub_Category, sum(Sales) as Total_Sales from joined_data_view group by 1,2,3 order by 1").show()

+-------+----------------+--------------------+------------------+
|Prod_id|Product_Category|Product_Sub_Category|       Total_Sales|
+-------+----------------+--------------------+------------------+
| Prod_1| OFFICE SUPPLIES|STORAGE & ORGANIZ...|1070182.6000000006|
|Prod_10|       FURNITURE|           BOOKCASES| 822652.0400000003|
|Prod_11|       FURNITURE|              TABLES|1896008.1420000014|
|Prod_12| OFFICE SUPPLIES|              LABELS| 38981.55000000002|
|Prod_13| OFFICE SUPPLIES| PENS & ART SUPPLIES|167107.21999999997|
|Prod_14|      TECHNOLOGY|     COPIERS AND FAX|1130361.2999999996|
|Prod_15|       FURNITURE|  CHAIRS & CHAIRMATS|        1761836.55|
|Prod_16| OFFICE SUPPLIES|SCISSORS, RULERS ...| 80996.30999999997|
|Prod_17|      TECHNOLOGY|     OFFICE MACHINES|2168697.1400000006|
| Prod_2| OFFICE SUPPLIES|          APPLIANCES| 736991.5399999997|
| Prod_3| OFFICE SUPPLIES|BINDERS AND BINDE...|1022957.5900000007|
| Prod_4|      TECHNOLOGY|TELEPHONES AND CO...| 1889313.801999

# Q8. Count the total number of unique customers in January and how many of them came back every month over the entire year in 2011.

In [30]:
spark.sql("Select Cust_id, Customer_Name, Customer_Segment, Order_Date from joined_data_view where Month(Order_Date) = 1 and Year(Order_Date) = 2011 group by 1,2,3,4 order by 1 ").show()

+---------+----------------+----------------+----------+
|  Cust_id|   Customer_Name|Customer_Segment|Order_Date|
+---------+----------------+----------------+----------+
|Cust_1010|   ROBERT MARLEY|       CORPORATE|2011-01-29|
|Cust_1012|     GUY PHONELY|       CORPORATE|2011-01-17|
|Cust_1016|     VIVEK GRADY|     HOME OFFICE|2011-01-11|
|Cust_1033|       RYAN AKIN|        CONSUMER|2011-01-08|
|Cust_1059|  TROY BLACKWELL|     HOME OFFICE|2011-01-18|
|Cust_1100|    TROY STAEBEL|  SMALL BUSINESS|2011-01-22|
|Cust_1146| PATRICK GARDNER|        CONSUMER|2011-01-09|
| Cust_115|    RALPH ARNETT|        CONSUMER|2011-01-04|
|Cust_1183|   TANJA NORVELL|        CONSUMER|2011-01-19|
|Cust_1184|LINDA SOUTHWORTH|        CONSUMER|2011-01-01|
|Cust_1193|   MICK CREBAGGA|     HOME OFFICE|2011-01-07|
|Cust_1195|     STEVEN WARD|        CONSUMER|2011-01-02|
|Cust_1195|     STEVEN WARD|        CONSUMER|2011-01-01|
|Cust_1206|  JILL STEVENSON|        CONSUMER|2011-01-21|
| Cust_121| SYLVIA FOULSTON|   

In [31]:
spark.sql("""
    SELECT Cust_id, Customer_Name, Customer_Segment, COUNT(DISTINCT MONTH(Order_Date)) AS distinct_months
    FROM joined_data_view
    WHERE YEAR(Order_Date) = 2011
    GROUP BY Cust_id, Customer_Name, Customer_Segment
    HAVING COUNT(DISTINCT MONTH(Order_Date)) = 4
    ORDER BY Cust_id
""").show()

+---------+----------------+----------------+---------------+
|  Cust_id|   Customer_Name|Customer_Segment|distinct_months|
+---------+----------------+----------------+---------------+
|Cust_1184|LINDA SOUTHWORTH|        CONSUMER|              4|
|Cust_1291|CINDY SCHNELLING|       CORPORATE|              4|
|Cust_1334|MAXWELL SCHWARTZ|       CORPORATE|              4|
|Cust_1337|   DENISE MONTON|     HOME OFFICE|              4|
|Cust_1338|LYCORIS SAUNDERS|     HOME OFFICE|              4|
| Cust_138|     JACK LEBRON|       CORPORATE|              4|
|Cust_1421| LIZ MACKENDRICK|       CORPORATE|              4|
|Cust_1707|  PAULINE WEBBER|        CONSUMER|              4|
| Cust_195|   SALLY KNUTSON|        CONSUMER|              4|
| Cust_478|   JAS O'CARROLL|       CORPORATE|              4|
| Cust_525|      DAN LAWERA|       CORPORATE|              4|
| Cust_939|     BEN WALLACE|     HOME OFFICE|              4|
+---------+----------------+----------------+---------------+



In [32]:
spark.sql("""
    SELECT Cust_id, Customer_Name, Customer_Segment, COUNT(DISTINCT MONTH(Order_Date)) AS distinct_months
    FROM joined_data_view
    WHERE YEAR(Order_Date) = 2011
    GROUP BY Cust_id, Customer_Name, Customer_Segment
    HAVING COUNT(DISTINCT MONTH(Order_Date)) = 12
    ORDER BY Cust_id
""").show()

+-------+-------------+----------------+---------------+
|Cust_id|Customer_Name|Customer_Segment|distinct_months|
+-------+-------------+----------------+---------------+
+-------+-------------+----------------+---------------+



In [33]:
# Create a temporary view of customers who ordered in January 2011
spark.sql("""
    CREATE OR REPLACE TEMP VIEW january_customers AS
    SELECT DISTINCT Cust_id
    FROM joined_data_view
    WHERE YEAR(Order_Date) = 2011 AND MONTH(Order_Date) = 1
""")

# Count the total unique customers in January
spark.sql("""
    SELECT COUNT(DISTINCT Cust_id) AS unique_customers_in_january
    FROM january_customers
""").show()

# Now, find customers who came back every month in 2011
customers_returned_every_month = spark.sql("""
    SELECT COUNT(DISTINCT j.Cust_id) AS customers_returned_every_month
    FROM january_customers j
    JOIN (
        SELECT Cust_id
        FROM joined_data_view
        WHERE YEAR(Order_Date) = 2011
        GROUP BY Cust_id
        HAVING COUNT(DISTINCT MONTH(Order_Date)) = 12
    ) r
    ON j.Cust_id = r.Cust_id
""")

customers_returned_every_month.show()


+---------------------------+
|unique_customers_in_january|
+---------------------------+
|                         99|
+---------------------------+

+------------------------------+
|customers_returned_every_month|
+------------------------------+
|                             0|
+------------------------------+



In [34]:
# Way - 2

jan_customers = df.filter(((F.year('Order_Date')) == 2011) & (F.month('Order_Date') == 1)).select("Cust_id").distinct()
jan_customers.show()

+---------+
|  Cust_id|
+---------+
|Cust_1033|
| Cust_214|
| Cust_538|
|Cust_1489|
|Cust_1306|
| Cust_375|
| Cust_559|
| Cust_993|
| Cust_232|
| Cust_514|
| Cust_521|
| Cust_287|
|Cust_1439|
|Cust_1012|
|Cust_1717|
| Cust_551|
| Cust_990|
|Cust_1010|
|Cust_1183|
|Cust_1276|
+---------+
only showing top 20 rows



In [35]:
print(jan_customers.count())

99


In [36]:
customers_returned_every_month = df.filter(F.year("Order_Date") == 2011).groupBy("Cust_id") \
    .agg(F.countDistinct(F.month("Order_Date")).alias("months_ordered")).filter(F.col("months_ordered") == 12) \
    .select("Cust_id").join(jan_customers, on="Cust_id", how="inner") \
    .agg(F.countDistinct("Cust_id").alias("customers_returned_every_month"))

customers_returned_every_month.show()

+------------------------------+
|customers_returned_every_month|
+------------------------------+
|                             0|
+------------------------------+



# Q: 9 Save the above Q:8 output as a count_month.json file.

In [38]:
customers_returned_every_month.write.json("count_month", mode = 'overwrite')

Py4JJavaError: An error occurred while calling o281.json.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:793)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1249)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1454)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:601)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:761)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:1972)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2014)
	at org.apache.hadoop.mapreduce.lib.output.FileOutputCommitter.getAllCommittedTaskPaths(FileO

In [ ]:
# df.write.json("joined_table")

In [40]:
df = df.toPandas()

In [41]:
df.to_json('output')